## Create environment

In [1]:
%%bash
set -e

cd /content
pwd

curl -fL https://micro.mamba.pm/api/micromamba/linux-64/latest -o /content/micromamba.tar.bz2
ls -lh /content/micromamba.tar.bz2

tar -xvjf /content/micromamba.tar.bz2 -C /usr/local/bin --strip-components=1 bin/micromamba

which micromamba
micromamba --version

/content
-rw-r--r-- 1 root root 6.6M May  6 14:29 /content/micromamba.tar.bz2
bin/micromamba
/usr/local/bin/micromamba
2.6.0


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:--  0:00:01 --:--:--     0
100  4091    0  4091    0     0   2345      0 --:--:--  0:00:01 --:--:--  2345
100 6725k  100 6725k    0     0  2557k      0  0:00:02  0:00:02 --:--:-- 8192k


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os

PROJECT_ROOT = "/content/drive/MyDrive/GWIT2"

# cache persistenti
HF_HOME = f"{PROJECT_ROOT}/.cache/huggingface"
PIP_CACHE_DIR = f"{PROJECT_ROOT}/.cache/pip"

# runtime locale
ENV_PATH = "/content/micromamba/envs/gwit"
LOCAL_ROOT = "/content/gwit_runtime"
LOCAL_LATENTS = f"{LOCAL_ROOT}/latents"

os.makedirs(PROJECT_ROOT, exist_ok=True)
os.makedirs(HF_HOME, exist_ok=True)
os.makedirs(PIP_CACHE_DIR, exist_ok=True)
os.makedirs(LOCAL_ROOT, exist_ok=True)
os.makedirs(LOCAL_LATENTS, exist_ok=True)

os.chdir(PROJECT_ROOT)

print("PROJECT_ROOT =", PROJECT_ROOT)
print("ENV_PATH     =", ENV_PATH)
print("HF_HOME      =", HF_HOME)

PROJECT_ROOT = /content/drive/MyDrive/GWIT2
ENV_PATH     = /content/micromamba/envs/gwit
HF_HOME      = /content/drive/MyDrive/GWIT2/.cache/huggingface


In [4]:
import os

os.environ["HF_HOME"] = HF_HOME
os.environ["HUGGINGFACE_HUB_CACHE"] = f"{HF_HOME}/hub"
os.environ["HF_DATASETS_CACHE"] = f"{HF_HOME}/datasets"
os.environ["PIP_CACHE_DIR"] = PIP_CACHE_DIR
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

print("HF_HOME =", os.environ["HF_HOME"])
print("HF_DATASETS_CACHE =", os.environ["HF_DATASETS_CACHE"])
print("PIP_CACHE_DIR =", os.environ["PIP_CACHE_DIR"])

HF_HOME = /content/drive/MyDrive/GWIT2/.cache/huggingface
HF_DATASETS_CACHE = /content/drive/MyDrive/GWIT2/.cache/huggingface/datasets
PIP_CACHE_DIR = /content/drive/MyDrive/GWIT2/.cache/pip


In [5]:
acc_dir = f"{HF_HOME}/accelerate"
os.makedirs(acc_dir, exist_ok=True)

config_text = """compute_environment: LOCAL_MACHINE
distributed_type: NO
machine_rank: 0
mixed_precision: fp16
num_machines: 1
num_processes: 1
use_cpu: false
"""

with open(f"{acc_dir}/default_config.yaml", "w") as f:
    f.write(config_text)

print("✔ accelerate default_config.yaml creato in", acc_dir)

✔ accelerate default_config.yaml creato in /content/drive/MyDrive/GWIT2/.cache/huggingface/accelerate


In [7]:

import os

if not os.path.isdir(ENV_PATH):
    print("Creo environment locale...")
    !micromamba create -y -p "{ENV_PATH}" python=3.9
    !micromamba run -p "{ENV_PATH}" pip install --upgrade pip
    !micromamba run -p "{ENV_PATH}" pip install ./diffusers/
    !micromamba run -p "{ENV_PATH}" pip install -r requirements.txt
    !micromamba run -p "{ENV_PATH}" pip install hf_transfer
else:
    print("Environment locale già presente in questa sessione.")

Creo environment locale...
[+] 0.0s
[+] 0.1s
Fetch Shard Index for conda-forge/linux-64                                                ✔ Done (0.2 sec)
Fetch Shard Index for conda-forge/noarch                                                  ✔ Done (0.1 sec)
Fetching and Parsing Packages' Shards                                                    ✔ Done (10.5 sec)
Using Cached Shard Index for conda-forge/linux-64                                                   ✔ Done
Using Cached Shard Index for conda-forge/noarch                                                     ✔ Done
Fetching and Parsing Packages' Shards                                                     ✔ Done (0.1 sec)

Resolving Environment                                                                     ✔ Done (0.5 sec)

Transaction

  Prefix: /content/micromamba/envs/gwit

  Updating specs:

   - python=3.9
   - pip


  Package               Version  Build                 Channel          Size
───────────────────────────

In [ ]:
DRIVE_LATENTS = f"{PROJECT_ROOT}/data/latents"
LOCAL_LATENTS = f"{LOCAL_ROOT}/latents"

!mkdir -p "{LOCAL_LATENTS}"
!rsync -ah --delete --info=progress2 "{DRIVE_LATENTS}/" "{LOCAL_LATENTS}/"

In [8]:
DRIVE_CLIP_EMBEDS = f"{PROJECT_ROOT}/data/clip_embeds_openclip_zebra"
LOCAL_CLIP_EMBEDS = f"{LOCAL_ROOT}/clip_embeds"

!mkdir -p "{LOCAL_CLIP_EMBEDS}"
!rsync -ah --delete --info=progress2 "{DRIVE_CLIP_EMBEDS}/" "{LOCAL_CLIP_EMBEDS}/"

         20.41G 100%   38.31MB/s    0:08:27 (xfr#24, to-chk=0/32)


## Precompute latents

In [ ]:
%cd "/content/drive/My Drive/GWIT2"

!micromamba run -n gwit python3 precompute_latents.py \
  --model_name Manojb/stable-diffusion-2-1-base \
  --dataset_name luigi-s/EEG_Image_CVPR_ALL_subj \
  --subjects 1 2 3 4 5 6 \
  --batch_size 16 \
  --save_every 25 \
  --output_root data/latents

/content/drive/My Drive/GWIT2
[INFO] Loading full HF pool: train + validation + test
Generating test split: 100% 1987/1987 [00:02<00:00, 763.57 examples/s]
Generating train split: 100% 7959/7959 [00:13<00:00, 598.56 examples/s]
Generating validation split: 100% 1994/1994 [00:07<00:00, 263.73 examples/s]
[INFO] Full pool size: 11940 samples
[INFO] Loading VAE...
config.json: 100% 553/553 [00:00<00:00, 344kB/s]
diffusion_pytorch_model.safetensors: 100% 335M/335M [00:09<00:00, 35.4MB/s]
[INFO] Processing subject 1
Filter: 100% 11940/11940 [01:17<00:00, 154.82 examples/s]
[INFO] Subject 1: 1980 samples in full pool
[INFO] Starting latent computation for subj1 with batch_size=16
subj1:  19% 24/124 [01:31<06:18,  3.78s/it][CHECKPOINT] subj1: saved 400 / 1980
subj1:  40% 49/124 [03:07<04:47,  3.84s/it][CHECKPOINT] subj1: saved 800 / 1980
subj1:  60% 74/124 [04:44<03:11,  3.83s/it][CHECKPOINT] subj1: saved 1200 / 1980
subj1:  80% 99/124 [06:21<01:36,  3.88s/it][CHECKPOINT] subj1: saved 1600 / 

## Precompute OpenCLIP embeddings

Image embeds

In [ ]:
!micromamba run -p "{ENV_PATH}" python precompute_openclip_embeds_zebra.py \
  --dataset_name luigi-s/EEG_Image_CVPR_ALL_subj \
  --subjects 1 2 3 4 5 6 \
  --image_column image \
  --cache_dir "{HF_HOME}" \
  --batch_size 16 \
  --output_root /content/drive/MyDrive/GWIT2/data/clip_embeds_openclip_zebra \
  --arch ViT-bigG-14 \
  --pretrained laion2b_s39b_b160k

[INFO] device = cuda
[INFO] Loading full HF pool for luigi-s/EEG_Image_CVPR_ALL_subj ...
[INFO] Total samples in full HF pool: 11940
[INFO] Loading FrozenOpenCLIPImageEmbedder...
[INFO] Dummy embedder output shape: (2, 256, 1664)
[INFO] Dummy token count T = 256
[INFO] Dummy embed dim   D = 1664
[INFO] subj1: 1980 samples
[INFO] subj1: creating new memmap with shape (1980, 256, 1664)
subj1:   0% 0/124 [00:00<?, ?it/s][INFO] subj1 first batch token shape: (16, 256, 1664) (B=16, T=256, D=1664)
subj1:   7% 9/124 [00:15<03:08,  1.64s/it][INFO] subj1: flushed memmap at 160/1980
subj1:  15% 19/124 [00:31<02:53,  1.65s/it][INFO] subj1: flushed memmap at 320/1980
subj1:  23% 29/124 [00:49<02:42,  1.71s/it][INFO] subj1: flushed memmap at 480/1980
subj1:  31% 39/124 [01:07<02:32,  1.79s/it][INFO] subj1: flushed memmap at 640/1980
subj1:  40% 49/124 [01:26<02:24,  1.92s/it][INFO] subj1: flushed memmap at 800/1980
subj1:  48% 59/124 [01:45<02:00,  1.85s/it][INFO] subj1: flushed memmap at 960/1980


Text embeds

In [ ]:
!micromamba run -p "{ENV_PATH}" python precompute_openclip_text_embeds_zebra.py \
  --dataset_name luigi-s/EEG_Image_CVPR_ALL_subj \
  --subjects 1 2 3 4 5 6 \
  --caption_column caption \
  --cache_dir "{HF_HOME}" \
  --batch_size 64 \
  --output_root /content/drive/MyDrive/GWIT2/data/clip_embeds_openclip_zebra \
  --arch ViT-bigG-14 \
  --pretrained laion2b_s39b_b160k

[INFO] device = cuda
[INFO] Loading full HF pool for luigi-s/EEG_Image_CVPR_ALL_subj ...
[INFO] Total samples in full HF pool: 11940
[INFO] Loading FrozenOpenCLIPEmbedder2...
[INFO] Dummy pooled text output shape: (2, 1280)
[INFO] Dummy pooled embed dim D = 1280
[INFO] subj1: 1980 samples
[INFO] subj1: creating new memmap with shape (1980, 1280)
subj1:   0% 0/31 [00:00<?, ?it/s][INFO] subj1 first batch pooled text shape: (64, 1280) (B=64, D=1280)
subj1:  61% 19/31 [00:33<00:23,  1.94s/it][INFO] subj1: flushed memmap at 1280/1980
subj1: 100% 31/31 [00:39<00:00,  1.29s/it]
[INFO] subj1: converting memmap to final .npy ...
[DONE] subj1 -> shape=(1980, 1280)
[DONE] saved: /content/drive/MyDrive/GWIT2/data/clip_embeds_openclip_zebra/luigi-s_EEG_Image_CVPR_ALL_subj/subj1/clip_text_embeds.npy
[INFO] subj2: 1992 samples
[INFO] subj2: creating new memmap with shape (1992, 1280)
subj2:   0% 0/32 [00:00<?, ?it/s][INFO] subj2 first batch pooled text shape: (64, 1280) (B=64, D=1280)
subj2:  59% 19/

## Baseline (GWIT reproduction)

### Start training (baseline)

In [ ]:
import wandb
wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: lorenzosoannini (lorenzosoannini-sapienza-universit-di-roma) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [ ]:
%cd "/content/drive/My Drive/GWIT2"

!micromamba run -n gwit accelerate launch train.py \
  --caption_from_classifier \
  --subject_num=4 \
  --pretrained_model_name_or_path=Manojb/stable-diffusion-2-1-base \
  --output_dir=output/BASELINE_CVPR_SUBJ4_CLASSIFIER_CAPTION_drop50_NewHope \
  --dataset_name=luigi-s/EEG_Image_CVPR_ALL_subj \
  --conditioning_image_column=conditioning_image \
  --image_column=image \
  --caption_column=caption \
  --learning_rate=1e-5 \
  --train_batch_size=6 \
  --num_train_epochs=50 \
  --checkpointing_steps=2000 \
  --validation_steps=500 \
  --enable_xformers_memory_efficient_attention \
  --report_to=wandb \
  --tracker_project_name=BASELINE_CVPR_SUBJ4_CLASSIFIER_CAPTION_drop50_NewHope \
  --data_root="/content/drive/My Drive/GWIT2/data" \
  --use_precomputed_latents \
  --latents_dir="data/luigi-s_EEG_Image_CVPR_ALL_subj_latents_train_subj4" \
  --drop_coarse_control_prob=0.5 \
  --log_drop_coarse_control \
  #--resume_from_checkpoint=checkpoint-8000

/content
2026-02-23 11:48:32,560 - INFO - Distributed environment: NO
Num processes: 1
Process index: 0
Local process index: 0
Device: cuda

Mixed precision type: fp16

tokenizer_config.json: 100% 807/807 [00:00<00:00, 138kB/s]
vocab.json: 1.06MB [00:00, 14.0MB/s]
merges.txt: 525kB [00:00, 42.3MB/s]
special_tokens_map.json: 100% 460/460 [00:00<00:00, 95.3kB/s]
/root/.local/share/mamba/envs/gwit/lib/python3.9/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
config.json: 100% 613/613 [00:00<00:00, 180kB/s]
You are using a model of type clip_text_model to instantiate a model of type . This is not supported for all configurations of models and can yield errors.
scheduler_config.js

### Generate images

In [ ]:
%cd "/content/drive/My Drive/GWIT2"

!micromamba run -n gwit \
  python generate_controlnet.py \
    --controlnet_path output/BASELINE_CVPR_SUBJ4_CLASSIFIER_CAPTION_drop50_NewHope \
    --pretrained_model_name_or_path Manojb/stable-diffusion-2-1-base \
    --caption \
    --subject 4 \
    --single_image_for_eval \
    --guess \
    --batch_size 4

/content
The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.
0it [00:00, ?it/s]
STO USANDO LA LIBRERIA GIUSTA
model_index.json: 100% 543/543 [00:00<00:00, 105kB/s]
Fetching 13 files:   0% 0/13 [00:00<?, ?it/s]
merges.txt: 0.00B [00:00, ?B/s]

preprocessor_config.json: 100% 342/342 [00:00<00:00, 33.7kB/s]



tokenizer_config.json:   0% 0.00/807 [00:00<?, ?B/s]

config.json:   0% 0.00/613 [00:00<?, ?B/s]



merges.txt: 525kB [00:00, 15.2MB/s]
tokenizer_config.json: 100% 807/807 [00:00<00:00, 38.2kB/s]


scheduler_config.json:   0% 0.00/346 [00:00<?, ?B/s]
config.json: 100% 613/613 [00:00<00:00, 22.9kB/s]
scheduler_config.json: 100% 346/346 [00:00<00:00, 27.5kB/s]
special_tokens_map.json: 100% 460/460 [00:00<00:00, 69.4kB/s]
vocab.json: 1.06MB [00:00, 29.6MB/s]

model.safetensors:   0% 0.00/1.36G [00:00<?, ?B/s]

### Evaluate

In [ ]:
%cd "/content/drive/My Drive/GWIT2"

!micromamba run -n gwit python evaluation/evaluate.py \
  --controlnet_path output/BASELINE_CVPR_SUBJ4_CLASSIFIER_CAPTION_drop50_NewHope \
  --batch_size 32 \
  --limit 4 \
  --guess \
  --GA

/content
Downloading: "https://download.pytorch.org/models/alexnet-owt-7be5be79.pth" to /root/.cache/torch/hub/checkpoints/alexnet-owt-7be5be79.pth
100% 233M/233M [00:01<00:00, 239MB/s]
/root/.local/share/mamba/envs/gwit/lib/python3.9/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/root/.local/share/mamba/envs/gwit/lib/python3.9/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=Inception_V3_Weights.IMAGENET1K_V1`. You can also use `weights=Inception_V3_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/inception_v3_google-0cc3c7bd.pth" to /root/.cache/torch/hub/checkpoints/inception_v3_go

## Cella di test

In [ ]:
import wandb
wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: lorenzosoannini (lorenzosoannini-sapienza-universit-di-roma) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [ ]:
!micromamba run -p "{ENV_PATH}" accelerate launch train.py \
  --pretrained_model_name_or_path Manojb/stable-diffusion-2-1-base \
  --dataset_name luigi-s/EEG_Image_CVPR_ALL_subj \
  --data_root /content/drive/MyDrive/GWIT2/data \
  --output_dir /content/drive/MyDrive/GWIT2/output/output_zebra_eeg_only_prior_000_lambda_subject_inv_1_v1 \
  --report_to wandb \
  --tracker_project_name output_zebra_eeg_only_prior_000_lambda_subject_inv_1_v1 \
  --train_subjects 1 2 3 4 5 \
  --val_subjects 1 2 3 4 5 \
  --test_subjects 6 \
  --val_ratio 0.1 \
  --split_seed 42 \
  --train_batch_size 6 \
  --val_batch_size 4 \
  --max_train_steps 6000 \
  --checkpointing_steps 6000 \
  --checkpoints_total_limit 3 \
  --console_log_every 20 \
  --mixed_precision fp16 \
  --enable_xformers_memory_efficient_attention \
  --learning_rate 1e-5 \
  --eeg_backbone_lr 1e-5 \
  --sife_lr 1e-5 \
  --recon_lr 1e-5 \
  --ssfe_lr 1e-5 \
  --prior_lr 1e-5 \
  --eeg_backbone_ckpt /content/drive/MyDrive/GWIT2/data/eegfeat_cvpr.pth \
  --caption_from_classifier \
  --use_precomputed_clip_embeds \
  --clip_embeds_dir "/content/gwit_runtime/clip_embeds" \
  --use_sife \
  --use_eeg_reconstruction \
  --use_ssfe \
  --use_prior \
  --train_eeg_only \
  --grl_lambda_sife 1.0 \
  --grl_lambda_ssfe 1.0 \
  --lambda_subject_inv 1.0 \
  --lambda_subject_spec 1.0 \
  --lambda_recon 0.5 \
  --lambda_ssfe 0.5 \
  --lambda_image_cls 1.0 \
  --lambda_image_dis 1.0 \
  --lambda_anchor_cls 0.5 \
  --lambda_anchor_visual 0.5 \
  --lambda_anchor_text 0.0 \
  --lambda_prior 1.0 \
  --ssfe_target_tokens 49 \
  --ssfe_out_dim 768 \
  --ssfe_adapter_type simple \
  --max_train_samples_per_subject 400 \
  --max_val_samples_per_subject 1

2026-03-31 10:59:42,462 - INFO - Distributed environment: NO
Num processes: 1
Process index: 0
Local process index: 0
Device: cuda

Mixed precision type: fp16

/content/micromamba/envs/gwit/lib/python3.9/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
You are using a model of type clip_text_model to instantiate a model of type . This is not supported for all configurations of models and can yield errors.
[AUTO] text_dim = 1024
2026-03-31 11:02:20,899 - INFO - Initializing ControlNet from UNet
STO USANDO LA LIBRERIA GIUSTA
[EEG DATASET] Loaded full HF pool: 11940 samples
[EEG DATASET] After subject index map [1, 2, 3, 4, 5]: 9949 samples
[EEG DATASET] Subject 1 | mode=train | 

In [ ]:
!micromamba run -p "{ENV_PATH}" accelerate launch train.py \
  --pretrained_model_name_or_path Manojb/stable-diffusion-2-1-base \
  --dataset_name luigi-s/EEG_Image_CVPR_ALL_subj \
  --data_root /content/drive/MyDrive/GWIT2/data \
  --output_dir /content/drive/MyDrive/GWIT2/output/output_zebra_eeg_only_prior_000_lambda_subject_inv_2_v1 \
  --report_to wandb \
  --tracker_project_name output_zebra_eeg_only_prior_000_lambda_subject_inv_2_v1 \
  --train_subjects 1 2 3 4 5 \
  --val_subjects 1 2 3 4 5 \
  --test_subjects 6 \
  --val_ratio 0.1 \
  --split_seed 42 \
  --train_batch_size 6 \
  --val_batch_size 4 \
  --max_train_steps 6000 \
  --checkpointing_steps 6000 \
  --checkpoints_total_limit 3 \
  --console_log_every 20 \
  --mixed_precision fp16 \
  --enable_xformers_memory_efficient_attention \
  --learning_rate 1e-5 \
  --eeg_backbone_lr 1e-5 \
  --sife_lr 1e-5 \
  --recon_lr 1e-5 \
  --ssfe_lr 1e-5 \
  --prior_lr 1e-5 \
  --eeg_backbone_ckpt /content/drive/MyDrive/GWIT2/data/eegfeat_cvpr.pth \
  --caption_from_classifier \
  --use_precomputed_clip_embeds \
  --clip_embeds_dir "/content/gwit_runtime/clip_embeds" \
  --use_sife \
  --use_eeg_reconstruction \
  --use_ssfe \
  --use_prior \
  --train_eeg_only \
  --grl_lambda_sife 1.0 \
  --grl_lambda_ssfe 1.0 \
  --lambda_subject_inv 2.0 \
  --lambda_subject_spec 1.0 \
  --lambda_recon 0.5 \
  --lambda_ssfe 0.5 \
  --lambda_image_cls 1.0 \
  --lambda_image_dis 1.0 \
  --lambda_anchor_cls 0.5 \
  --lambda_anchor_visual 0.5 \
  --lambda_anchor_text 0.0 \
  --lambda_prior 0.0 \
  --ssfe_target_tokens 49 \
  --ssfe_out_dim 768 \
  --ssfe_adapter_type simple \
  --max_train_samples_per_subject 400 \
  --max_val_samples_per_subject 1

2026-03-31 11:32:11,478 - INFO - Distributed environment: NO
Num processes: 1
Process index: 0
Local process index: 0
Device: cuda

Mixed precision type: fp16

/content/micromamba/envs/gwit/lib/python3.9/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
You are using a model of type clip_text_model to instantiate a model of type . This is not supported for all configurations of models and can yield errors.
[AUTO] text_dim = 1024
2026-03-31 11:32:21,117 - INFO - Initializing ControlNet from UNet
STO USANDO LA LIBRERIA GIUSTA
[EEG DATASET] Loaded full HF pool: 11940 samples
[EEG DATASET] After subject index map [1, 2, 3, 4, 5]: 9949 samples
[EEG DATASET] Subject 1 | mode=train | 

In [ ]:
!micromamba run -p "{ENV_PATH}" accelerate launch train.py \
  --pretrained_model_name_or_path Manojb/stable-diffusion-2-1-base \
  --dataset_name luigi-s/EEG_Image_CVPR_ALL_subj \
  --data_root /content/drive/MyDrive/GWIT2/data \
  --output_dir /content/drive/MyDrive/GWIT2/output/output_zebra_eeg_only_prior_000_lambda_subject_inv_1_grl_lambda_sife_2_v1 \
  --report_to wandb \
  --tracker_project_name output_zebra_eeg_only_prior_000_lambda_subject_inv_1_grl_lambda_sife_2_v1 \
  --train_subjects 1 2 3 4 5 \
  --val_subjects 1 2 3 4 5 \
  --test_subjects 6 \
  --val_ratio 0.1 \
  --split_seed 42 \
  --train_batch_size 6 \
  --val_batch_size 4 \
  --max_train_steps 6000 \
  --checkpointing_steps 6000 \
  --checkpoints_total_limit 3 \
  --console_log_every 20 \
  --mixed_precision fp16 \
  --enable_xformers_memory_efficient_attention \
  --learning_rate 1e-5 \
  --eeg_backbone_lr 1e-5 \
  --sife_lr 1e-5 \
  --recon_lr 1e-5 \
  --ssfe_lr 1e-5 \
  --prior_lr 1e-5 \
  --eeg_backbone_ckpt /content/drive/MyDrive/GWIT2/data/eegfeat_cvpr.pth \
  --caption_from_classifier \
  --use_precomputed_clip_embeds \
  --clip_embeds_dir "/content/gwit_runtime/clip_embeds" \
  --use_sife \
  --use_eeg_reconstruction \
  --use_ssfe \
  --use_prior \
  --train_eeg_only \
  --grl_lambda_sife 2.0 \
  --grl_lambda_ssfe 1.0 \
  --lambda_subject_inv 1.0 \
  --lambda_subject_spec 1.0 \
  --lambda_recon 0.5 \
  --lambda_ssfe 0.5 \
  --lambda_image_cls 1.0 \
  --lambda_image_dis 1.0 \
  --lambda_anchor_cls 0.5 \
  --lambda_anchor_visual 0.5 \
  --lambda_anchor_text 0.0 \
  --lambda_prior 0.0 \
  --ssfe_target_tokens 49 \
  --ssfe_out_dim 768 \
  --ssfe_adapter_type simple \
  --max_train_samples_per_subject 400 \
  --max_val_samples_per_subject 1

2026-03-31 12:00:02,374 - INFO - Distributed environment: NO
Num processes: 1
Process index: 0
Local process index: 0
Device: cuda

Mixed precision type: fp16

/content/micromamba/envs/gwit/lib/python3.9/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
You are using a model of type clip_text_model to instantiate a model of type . This is not supported for all configurations of models and can yield errors.
[AUTO] text_dim = 1024
2026-03-31 12:00:14,913 - INFO - Initializing ControlNet from UNet
STO USANDO LA LIBRERIA GIUSTA
[EEG DATASET] Loaded full HF pool: 11940 samples
[EEG DATASET] After subject index map [1, 2, 3, 4, 5]: 9949 samples
[EEG DATASET] Subject 1 | mode=train | 

In [ ]:
!micromamba run -p "{ENV_PATH}" accelerate launch train.py \
  --pretrained_model_name_or_path Manojb/stable-diffusion-2-1-base \
  --dataset_name luigi-s/EEG_Image_CVPR_ALL_subj \
  --data_root /content/drive/MyDrive/GWIT2/data \
  --output_dir /content/drive/MyDrive/GWIT2/output/output_zebra_eeg_only_prior_100_lambda_subject_inv_1_grl_lambda_sife_2_v1 \
  --report_to wandb \
  --tracker_project_name output_zebra_eeg_only_prior_100_lambda_subject_inv_1_grl_lambda_sife_2_v1 \
  --train_subjects 1 2 3 4 5 \
  --val_subjects 1 2 3 4 5 \
  --test_subjects 6 \
  --val_ratio 0.1 \
  --split_seed 42 \
  --train_batch_size 6 \
  --val_batch_size 4 \
  --max_train_steps 6000 \
  --checkpointing_steps 6000 \
  --checkpoints_total_limit 3 \
  --console_log_every 20 \
  --mixed_precision fp16 \
  --enable_xformers_memory_efficient_attention \
  --learning_rate 1e-5 \
  --eeg_backbone_lr 1e-5 \
  --sife_lr 1e-5 \
  --recon_lr 1e-5 \
  --ssfe_lr 1e-5 \
  --prior_lr 1e-5 \
  --eeg_backbone_ckpt /content/drive/MyDrive/GWIT2/data/eegfeat_cvpr.pth \
  --caption_from_classifier \
  --use_precomputed_clip_embeds \
  --clip_embeds_dir "/content/gwit_runtime/clip_embeds" \
  --use_sife \
  --use_eeg_reconstruction \
  --use_ssfe \
  --use_prior \
  --train_eeg_only \
  --grl_lambda_sife 2.0 \
  --grl_lambda_ssfe 1.0 \
  --lambda_subject_inv 1.0 \
  --lambda_subject_spec 1.0 \
  --lambda_recon 0.5 \
  --lambda_ssfe 0.5 \
  --lambda_image_cls 1.0 \
  --lambda_image_dis 1.0 \
  --lambda_anchor_cls 0.5 \
  --lambda_anchor_visual 0.5 \
  --lambda_anchor_text 0.0 \
  --lambda_prior 1.0 \
  --ssfe_target_tokens 49 \
  --ssfe_out_dim 768 \
  --ssfe_adapter_type simple \
  --max_train_samples_per_subject 400 \
  --max_val_samples_per_subject 1

2026-04-01 16:48:01,766 - INFO - Distributed environment: NO
Num processes: 1
Process index: 0
Local process index: 0
Device: cuda

Mixed precision type: fp16

/content/micromamba/envs/gwit/lib/python3.9/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
You are using a model of type clip_text_model to instantiate a model of type . This is not supported for all configurations of models and can yield errors.
[AUTO] text_dim = 1024
2026-04-01 16:50:21,282 - INFO - Initializing ControlNet from UNet
STO USANDO LA LIBRERIA GIUSTA
[EEG DATASET] Loaded full HF pool: 11940 samples
[EEG DATASET] After subject index map [1, 2, 3, 4, 5]: 9949 samples
[EEG DATASET] Subject 1 | mode=train | 

In [ ]:
!micromamba run -p "{ENV_PATH}" accelerate launch train.py \
  --pretrained_model_name_or_path Manojb/stable-diffusion-2-1-base \
  --dataset_name luigi-s/EEG_Image_CVPR_ALL_subj \
  --data_root /content/drive/MyDrive/GWIT2/data \
  --output_dir /content/drive/MyDrive/GWIT2/output/loss_anchor_visual_bs64_zebra_v2 \
  --report_to wandb \
  --tracker_project_name loss_anchor_visual_bs64_zebra_v2 \
  --train_subjects 1 2 3 4 5 \
  --val_subjects 1 2 3 4 5 \
  --test_subjects 6 \
  --val_ratio 0.1 \
  --split_seed 42 \
  --seed 42 \
  --train_batch_size 64 \
  --val_batch_size 64 \
  --validation_steps 500 \
  --max_train_steps 6000 \
  --checkpointing_steps 1000 \
  --checkpoints_total_limit 3 \
  --console_log_every 20 \
  --mixed_precision fp16 \
  --enable_xformers_memory_efficient_attention \
  --learning_rate 1e-5 \
  --eeg_backbone_lr 1e-5 \
  --sife_lr 1e-5 \
  --recon_lr 1e-5 \
  --ssfe_lr 1e-5 \
  --prior_lr 1e-5 \
  --eeg_backbone_ckpt /content/drive/MyDrive/GWIT2/data/eegfeat_cvpr.pth \
  --caption_from_classifier \
  --use_precomputed_clip_embeds \
  --clip_embeds_dir "/content/gwit_runtime/clip_embeds" \
  --use_sife \
  --use_eeg_reconstruction \
  --use_ssfe \
  --use_prior \
  --train_eeg_only \
  --grl_lambda_sife 2.0 \
  --grl_lambda_ssfe 1.0 \
  --lambda_subject_inv 1.0 \
  --lambda_subject_spec 1.0 \
  --lambda_recon 0.5 \
  --lambda_ssfe 1.0 \
  --lambda_image_cls 1.0 \
  --lambda_image_dis 1.0 \
  --lambda_anchor_cls 0.5 \
  --lambda_anchor_visual 2.0 \
  --lambda_anchor_text 0.0 \
  --lambda_prior 1.0 \
  --ssfe_target_tokens 49 \
  --ssfe_out_dim 768 \
  --ssfe_adapter_type zebra_like \
  --max_train_samples_per_subject 400 \
  --max_val_samples_per_subject 40

2026-04-04 09:07:33,697 - INFO - Distributed environment: NO
Num processes: 1
Process index: 0
Local process index: 0
Device: cuda

Mixed precision type: fp16

/content/micromamba/envs/gwit/lib/python3.9/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
You are using a model of type clip_text_model to instantiate a model of type . This is not supported for all configurations of models and can yield errors.
[AUTO] text_dim = 1024
2026-04-04 09:09:34,040 - INFO - Initializing ControlNet from UNet
STO USANDO LA LIBRERIA GIUSTA
[EEG DATASET] Loaded full HF pool: 11940 samples
[EEG DATASET] After subject index map [1, 2, 3, 4, 5]: 9949 samples
[EEG DATASET] Subject 1 | mode=train | 

## Run 7

In [ ]:
!micromamba run -p "{ENV_PATH}" accelerate launch train.py \
  --pretrained_model_name_or_path Manojb/stable-diffusion-2-1-base \
  --dataset_name luigi-s/EEG_Image_CVPR_ALL_subj \
  --data_root /content/drive/MyDrive/GWIT2/data \
  --output_dir /content/drive/MyDrive/GWIT2/output/loss_anchor_visual_and_anchor_text_bs32_zebra_v1 \
  --report_to wandb \
  --tracker_project_name loss_anchor_visual_and_anchor_text_bs32_zebra_v1 \
  --train_subjects 1 2 3 4 5 \
  --val_subjects 1 2 3 4 5 \
  --test_subjects 6 \
  --val_ratio 0.1 \
  --split_seed 42 \
  --seed 42 \
  --train_batch_size 32 \
  --val_batch_size 32 \
  --validation_steps 500 \
  --checkpointing_steps 2000 \
  --console_log_every 200 \
  --mixed_precision fp16 \
  --enable_xformers_memory_efficient_attention \
  --learning_rate 1e-5 \
  --eeg_backbone_lr 1e-5 \
  --sife_lr 1e-5 \
  --recon_lr 1e-5 \
  --ssfe_lr 1e-5 \
  --prior_lr 1e-5 \
  --eeg_backbone_ckpt /content/drive/MyDrive/GWIT2/data/eegfeat_cvpr.pth \
  --caption_from_classifier \
  --use_precomputed_clip_embeds \
  --clip_embeds_dir "/content/gwit_runtime/clip_embeds" \
  --use_sife \
  --use_eeg_reconstruction \
  --use_ssfe \
  --use_prior \
  --train_eeg_only \
  --grl_lambda_sife 2.0 \
  --grl_lambda_ssfe 1.0 \
  --lambda_subject_inv 1.0 \
  --lambda_subject_spec 1.0 \
  --lambda_recon 0.5 \
  --lambda_ssfe 1.0 \
  --lambda_image_cls 1.0 \
  --lambda_image_dis 1.0 \
  --lambda_anchor_cls 0.5 \
  --lambda_anchor_visual 2.0 \
  --lambda_anchor_text 1.0 \
  --lambda_prior 1.0 \
  --ssfe_target_tokens 49 \
  --ssfe_out_dim 768 \
  --ssfe_adapter_type zebra_like \
  --max_train_samples_per_subject 400 \
  --max_val_samples_per_subject 40 \
  --num_train_epochs 500 \
  --resume_from_checkpoint=checkpoint-66000

2026-04-13 16:28:14,082 - INFO - Distributed environment: NO
Num processes: 1
Process index: 0
Local process index: 0
Device: cuda

Mixed precision type: fp16

/content/micromamba/envs/gwit/lib/python3.9/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
You are using a model of type clip_text_model to instantiate a model of type . This is not supported for all configurations of models and can yield errors.
[AUTO] text_dim = 1024
[ANCHOR] CLIP text dim=512 | model=openai/clip-vit-base-patch32
2026-04-13 16:29:34,608 - INFO - Initializing ControlNet from UNet
STO USANDO LA LIBRERIA GIUSTA
[EEG DATASET] Loaded full HF pool: 11940 samples
[EEG DATASET] After subject index map [1, 2

### Eval prior in CLIP space

In [ ]:
!micromamba run -p "{ENV_PATH}" python -m evaluation.infer_prior_clip \
  --output_dir /content/drive/MyDrive/GWIT2/output/loss_anchor_visual_and_anchor_text_bs32_zebra_v1 \
  --dataset_name luigi-s/EEG_Image_CVPR_ALL_subj \
  --data_root /content/drive/MyDrive/GWIT2/data \
  --clip_embeds_dir /content/gwit_runtime/clip_embeds \
  --pretrained_model_name_or_path Manojb/stable-diffusion-2-1-base \
  --train_subjects 1 2 3 4 5 \
  --val_subjects 1 2 3 4 5 \
  --test_subjects 6 \
  --test_batch_size 128 \
  --prior_inference_timesteps 20 \
  --prior_cond_scale 1.0

[INFO] device = cuda
[INFO] output_dir = /content/drive/MyDrive/GWIT2/output/loss_anchor_visual_and_anchor_text_bs32_zebra_v1
/content/micromamba/envs/gwit/lib/python3.9/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
[EEG DATASET] Loaded full HF pool: 11940 samples
[EEG DATASET] After subject index map [6]: 1991 samples
[EEG DATASET] Subject 6 | mode=test | kept 1991/1991 | cap=None
[EEG DATASET] Loaded CLIP embeds for subj6: shape=(1991, 49, 768) | mode=sequence
[EEG DATASET] Final dataset | mode=test | subjects=[6] | total=1991
[EEG DATASET] CLIP embedding mode: sequence-level
[INFO] test samples = 1991
[INFO] first clip target shape = (49, 768)
100% 16/16 [02:15<00:00,  8

In [ ]:
!micromamba run -p "{ENV_PATH}" python -m evaluation.eval_prior_clip \
  --inference_dir /content/drive/MyDrive/GWIT2/output/loss_anchor_visual_and_anchor_text_bs32_zebra_v1/prior_clip_inference \
  --topk 1 5 10

==== Prior CLIP-space metrics ====
num_samples: 1991
num_tokens: 49
embed_dim: 768
cosine_diag_mean: 0.754321
tokenwise_cosine_mean: 0.754293
mse: 0.000561
two_way_identification: 0.499999
sim_diag_mean: 0.754321
sim_diag_std: 0.020688
sim_diag_min: 0.684852
sim_diag_max: 0.808618
sim_offdiag_mean: 0.754321
sim_offdiag_std: 0.020688
retrieval_fwd_top1: 0.000502
retrieval_fwd_top5: 0.002511
retrieval_fwd_top10: 0.005023
retrieval_bwd_top1: 0.000000
retrieval_bwd_top5: 0.002009
retrieval_bwd_top10: 0.004018
inference_meta:
  pred_shape: [1991, 49, 768]
  gt_shape: [1991, 49, 768]
  Fs_shape: [1991, 49, 768]
  subjects_shape: [1991]
  labels_shape: [1991]
  prior_inference_timesteps: 20
  prior_cond_scale: 1.0
  dataset_name: luigi-s/EEG_Image_CVPR_ALL_subj
  test_subjects: [6]
[DONE] Metrics saved to /content/drive/MyDrive/GWIT2/output/loss_anchor_visual_and_anchor_text_bs32_zebra_v1/prior_clip_inference/prior_clip_metrics.json


## Run 8

In [ ]:
import wandb
wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 3


wandb: You chose "Don't visualize my results"
wandb: Using W&B in offline mode.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


False

In [ ]:
!micromamba run -p "{ENV_PATH}" accelerate launch train.py \
  --pretrained_model_name_or_path Manojb/stable-diffusion-2-1-base \
  --dataset_name luigi-s/EEG_Image_CVPR_ALL_subj \
  --data_root /content/drive/MyDrive/GWIT2/data \
  --output_dir /content/drive/MyDrive/GWIT2/output/bs128_zebralike_fulldataset_v1 \
  --report_to wandb \
  --tracker_project_name bs128_zebralike_fulldataset_v1 \
  --train_subjects 1 2 3 4 5 \
  --val_subjects 1 2 3 4 5 \
  --test_subjects 6 \
  --val_ratio 0.1 \
  --split_seed 42 \
  --seed 42 \
  --train_batch_size 16 \
  --val_batch_size 16 \
  --gradient_accumulation_steps 8 \
  --gradient_checkpointing_sife \
  --gradient_checkpointing_ssfe \
  --gradient_checkpointing_prior \
  --validation_steps 500 \
  --checkpointing_steps 2000 \
  --console_log_every 200 \
  --mixed_precision fp16 \
  --enable_xformers_memory_efficient_attention \
  --learning_rate 1e-4 \
  --eeg_backbone_lr 1e-4 \
  --sife_lr 1e-4 \
  --recon_lr 1e-4 \
  --ssfe_lr 1e-4 \
  --prior_lr 1e-4 \
  --eeg_backbone_ckpt /content/drive/MyDrive/GWIT2/data/eegfeat_cvpr.pth \
  --use_precomputed_clip_embeds \
  --clip_embeds_dir "/content/gwit_runtime/clip_embeds" \
  --use_sife \
  --use_eeg_reconstruction \
  --use_ssfe \
  --use_prior \
  --train_eeg_only \
  --grl_lambda_sife 1.0 \
  --grl_lambda_ssfe 1.0 \
  --lambda_subject_inv 1.0 \
  --lambda_subject_spec 1.0 \
  --lambda_recon 0.5 \
  --lambda_ssfe 1.0 \
  --lambda_image_cls 1.0 \
  --lambda_image_dis 1.0 \
  --lambda_anchor_cls 0.5 \
  --lambda_anchor_visual 0.5 \
  --lambda_anchor_text 0.25 \
  --lambda_prior 30.0 \
  --ssfe_adapter_type zebra_like \
  --num_train_epochs 60 \
  #--resume_from_checkpoint=checkpoint-66000

2026-04-16 17:27:35,107 - INFO - Distributed environment: NO
Num processes: 1
Process index: 0
Local process index: 0
Device: cuda

Mixed precision type: fp16

/content/micromamba/envs/gwit/lib/python3.9/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
You are using a model of type clip_text_model to instantiate a model of type . This is not supported for all configurations of models and can yield errors.
[AUTO] text_dim = 1024
2026-04-16 17:29:30,112 - INFO - Initializing ControlNet from UNet
STO USANDO LA LIBRERIA GIUSTA
[EEG DATASET] Loaded full HF pool: 11940 samples
[EEG DATASET] After subject index map [1, 2, 3, 4, 5]: 9949 samples
[EEG DATASET] Subject 1 | mode=train | 

## Run 9

In [6]:
import wandb
wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: lorenzosoannini (lorenzosoannini-sapienza-universit-di-roma) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

### Stage 1 — EEG backbone + SIFE + reconstruction

In [ ]:
!micromamba run -p "{ENV_PATH}" accelerate launch train.py \
  --pretrained_model_name_or_path Manojb/stable-diffusion-2-1-base \
  --dataset_name luigi-s/EEG_Image_CVPR_ALL_subj \
  --data_root /content/drive/MyDrive/GWIT2/data \
  --output_dir /content/drive/MyDrive/GWIT2/output/zebra_stage1 \
  --report_to wandb \
  --tracker_project_name zebra_stage1 \
  --train_subjects 1 2 3 4 5 \
  --val_subjects 1 2 3 4 5 \
  --test_subjects 6 \
  --val_ratio 0.1 \
  --split_seed 42 \
  --seed 42 \
  --train_batch_size 128 \
  --val_batch_size 128 \
  --gradient_accumulation_steps 1 \
  --validation_steps 500 \
  --checkpointing_steps 2000 \
  --console_log_every 200 \
  --mixed_precision fp16 \
  --learning_rate 1e-4 \
  --eeg_backbone_lr 1e-4 \
  --sife_lr 1e-4 \
  --recon_lr 1e-4 \
  --eeg_backbone_ckpt /content/drive/MyDrive/GWIT2/data/eegfeat_cvpr.pth \
  --use_sife \
  --use_eeg_reconstruction \
  --train_eeg_only \
  --training_stage stage1 \
  --grl_lambda_sife 1.0 \
  --lambda_subject_inv 1.0 \
  --lambda_subject_spec 1.0 \
  --lambda_recon 0.5 \
  --num_train_epochs 60

2026-04-17 09:56:25,975 - INFO - Distributed environment: NO
Num processes: 1
Process index: 0
Local process index: 0
Device: cuda

Mixed precision type: fp16

/content/micromamba/envs/gwit/lib/python3.9/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
[EEG DATASET] Loaded full HF pool: 11940 samples
[EEG DATASET] After subject index map [1, 2, 3, 4, 5]: 9949 samples
[EEG DATASET] Subject 1 | mode=train | kept 1782/1980 | cap=None
[EEG DATASET] Subject 2 | mode=train | kept 1793/1992 | cap=None
[EEG DATASET] Subject 3 | mode=train | kept 1793/1992 | cap=None
[EEG DATASET] Subject 4 | mode=train | kept 1795/1994 | cap=None
[EEG DATASET] Subject 5 | mode=train | kept 1792/1991 |

### Stage 2 — SSFE

In [ ]:
!micromamba run -p "{ENV_PATH}" accelerate launch train.py \
  --pretrained_model_name_or_path Manojb/stable-diffusion-2-1-base \
  --dataset_name luigi-s/EEG_Image_CVPR_ALL_subj \
  --data_root /content/drive/MyDrive/GWIT2/data \
  --output_dir /content/drive/MyDrive/GWIT2/output/zebra_stage2 \
  --report_to wandb \
  --tracker_project_name zebra_stage2 \
  --train_subjects 1 2 3 4 5 \
  --val_subjects 1 2 3 4 5 \
  --test_subjects 6 \
  --val_ratio 0.1 \
  --split_seed 42 \
  --seed 42 \
  --train_batch_size 128 \
  --val_batch_size 128 \
  --gradient_accumulation_steps 1 \
  --validation_steps 500 \
  --checkpointing_steps 500 \
  --console_log_every 200 \
  --mixed_precision fp16 \
  --learning_rate 1e-4 \
  --ssfe_lr 1e-4 \
  --eeg_backbone_ckpt /content/drive/MyDrive/GWIT2/output/zebra_stage1/eeg_backbone.pt \
  --load_sife_path /content/drive/MyDrive/GWIT2/output/zebra_stage1/sife.pt \
  --use_precomputed_clip_embeds \
  --clip_embeds_dir /content/gwit_runtime/clip_embeds \
  --use_sife \
  --use_ssfe \
  --train_eeg_only \
  --training_stage stage2 \
  --ssfe_adapter_type zebra_like \
  --grl_lambda_ssfe 1.0 \
  --lambda_ssfe 1.0 \
  --lambda_image_cls 1.0 \
  --lambda_image_dis 1.0 \
  --lambda_anchor_cls 0.5 \
  --lambda_anchor_visual 0.5 \
  --lambda_anchor_text 0.25 \
  --num_train_epochs 120 \
  --resume_from_checkpoint=checkpoint-4000

2026-04-28 18:38:04,993 - INFO - Distributed environment: NO
Num processes: 1
Process index: 0
Local process index: 0
Device: cuda

Mixed precision type: fp16

/content/micromamba/envs/gwit/lib/python3.9/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
[EEG DATASET] Loaded full HF pool: 11940 samples
[EEG DATASET] After subject index map [1, 2, 3, 4, 5]: 9949 samples
[EEG DATASET] Subject 1 | mode=train | kept 1782/1980 | cap=None
[EEG DATASET] Loaded CLIP image embeds for subj1: shape=(1980, 256, 1664) | mode=sequence
[EEG DATASET] Loaded CLIP text embeds for subj1: shape=(1980, 1280)
[EEG DATASET] Subject 2 | mode=train | kept 1793/1992 | cap=None
[EEG DATASET] Loaded CLIP i

### Stage 3 — Prior

In [ ]:
!micromamba run -p "{ENV_PATH}" accelerate launch train.py \
  --pretrained_model_name_or_path Manojb/stable-diffusion-2-1-base \
  --dataset_name luigi-s/EEG_Image_CVPR_ALL_subj \
  --data_root /content/drive/MyDrive/GWIT2/data \
  --output_dir /content/drive/MyDrive/GWIT2/output/zebra_stage3_v3 \
  --report_to wandb \
  --tracker_project_name zebra_stage3_v3 \
  --train_subjects 1 2 3 4 5 \
  --val_subjects 1 2 3 4 5 \
  --test_subjects 6 \
  --val_ratio 0.1 \
  --split_seed 42 \
  --seed 42 \
  --train_batch_size 8 \
  --val_batch_size 8 \
  --gradient_accumulation_steps 16 \
  --validation_steps 200 \
  --checkpointing_steps 150 \
  --console_log_every 100 \
  --mixed_precision fp16 \
  --learning_rate 1e-4 \
  --prior_lr 1e-4 \
  --eeg_backbone_ckpt /content/drive/MyDrive/GWIT2/output/zebra_stage1/eeg_backbone.pt \
  --load_sife_path /content/drive/MyDrive/GWIT2/output/zebra_stage1/sife.pt \
  --load_ssfe_path /content/drive/MyDrive/GWIT2/output/zebra_stage2/ssfe_projector.pt \
  --use_precomputed_clip_embeds \
  --clip_embeds_dir /content/gwit_runtime/clip_embeds \
  --use_sife \
  --use_ssfe \
  --use_prior \
  --train_eeg_only \
  --training_stage stage3 \
  --lambda_prior 1.0 \
  --num_train_epochs 200 \
  --resume_from_checkpoint=checkpoint-10800

2026-05-06 15:00:01,300 - INFO - Distributed environment: NO
Num processes: 1
Process index: 0
Local process index: 0
Device: cuda

Mixed precision type: fp16

/content/micromamba/envs/gwit/lib/python3.9/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
[EEG DATASET] Loaded full HF pool: 11940 samples
[EEG DATASET] After subject index map [1, 2, 3, 4, 5]: 9949 samples
[EEG DATASET] Subject 1 | mode=train | kept 1782/1980 | cap=None
[EEG DATASET] Loaded CLIP image embeds for subj1: shape=(1980, 256, 1664) | mode=sequence
[EEG DATASET] Loaded CLIP text embeds for subj1: shape=(1980, 1280)
[EEG DATASET] Subject 2 | mode=train | kept 1793/1992 | cap=None
[EEG DATASET] Loaded CLIP i

### Generate images

In [ ]:
!MPLBACKEND=Agg micromamba run -p "{ENV_PATH}" python recon.py \
  --pretrained_model_name_or_path Manojb/stable-diffusion-2-1-base \
  --dataset_name luigi-s/EEG_Image_CVPR_ALL_subj \
  --data_root /content/drive/MyDrive/GWIT2/data \
  --model_dir /content/drive/MyDrive/GWIT2/output/zebra_stage3 \
  --output_dir /content/drive/MyDrive/GWIT2/output/zebra_stage3_recon_test_subj6 \
  --test_subjects 6 \
  --batch_size 32 \
  --prior_inference_steps 20 \
  --prior_cond_scale 1.0 \
  --num_samples_per_image 1 \
  --save_vis \
  --save_pt \
  --unclip_ckpt /content/drive/MyDrive/GWIT2/data/unclip6_epoch0_step110000.ckpt

Fase A

In [ ]:
!micromamba run -p "{ENV_PATH}" python recon_stageA_tokens.py \
  --pretrained_model_name_or_path Manojb/stable-diffusion-2-1-base \
  --dataset_name luigi-s/EEG_Image_CVPR_ALL_subj \
  --data_root /content/drive/MyDrive/GWIT2/data \
  --model_dir /content/drive/MyDrive/GWIT2/output/zebra_stage3_v2 \
  --output_dir /content/drive/MyDrive/GWIT2/output/recon_stageA_subj6 \
  --tmp_dir /content/gwit_runtime/recon_stageA_tmp \
  --test_subjects 6 \
  --batch_size 64 \
  --prior_inference_steps 20 \
  --prior_cond_scale 1.0 \
  --save_gts \
  --resume

/content/micromamba/envs/gwit/lib/python3.9/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
[EEG DATASET] Loaded full HF pool: 11940 samples
[EEG DATASET] After subject index map [6]: 1991 samples
[EEG DATASET] Subject 6 | mode=test | kept 1991/1991 | cap=None
[EEG DATASET] Final dataset | mode=test | subjects=[6] | total=1991
[EEG Backbone Load]
Source: /content/drive/MyDrive/GWIT2/output/zebra_stage3_v2/eeg_backbone.pt
Missing keys: []
Unexpected keys: []
[RESUME] found 0 completed shard(s)
[RAM] before loop used=3.51 GB | avail=9.16 GB
Stage A: generating prior tokens:   0% 0/32 [00:00<?, ?it/s][RAM] after shard 00000 used=3.80 GB | avail=8.87 GB
Stage A: generating prior 

Fase B

In [ ]:
!MPLBACKEND=Agg micromamba run -p "{ENV_PATH}" python recon_stageB_unclip.py \
  --stageA_dir /content/drive/MyDrive/GWIT2/output/recon_stageA_subj6 \
  --output_dir /content/drive/MyDrive/GWIT2/output/recon_stageB_subj6 \
  --unclip_ckpt /content/drive/MyDrive/GWIT2/data/unclip6_epoch0_step110000_filtered_for_recon.safetensors \
  --unclip_config /content/drive/MyDrive/GWIT2/models/generative_models/configs/unclip6.yaml \
  --num_samples_per_image 1 \
  --decode_batch_size 1 \
  --model_dtype fp32 \
  --resume \
  --resume_minibatch \
  --save_final_manifest \
  --save_vis \
  --log_ram

/content/micromamba/envs/gwit/lib/python3.9/site-packages/lightning_fabric/__init__.py:29: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __import__("pkg_resources").declare_namespace(__name__)
No SDP backend available, likely because you are running in pytorch versions < 2.0. In fact, you are using PyTorch 1.13.1+cu117. You might want to consider upgrading.
[RAM] before prepare_unclip | used=1.71 GB | avail=10.96 GB
[RAM] before OmegaConf.load | used=1.71 GB | avail=10.96 GB
[RAM] before MinimalUnclipEngine init | used=1.72 GB | avail=10.95 GB
No SDP backend available, likely because you are running in pytorch versions < 2.0. In fact, you are using PyTorch 1.13.1+cu117. You might want to consider upgrading.
SpatialTransformer: Found context dims [1664] of depth 1, which does not match the sp

### DEBUG SECTION

In [ ]:
!MPLBACKEND=Agg micromamba run -p "{ENV_PATH}" python debug_decode_raw_gt_clip_tokens.py \
  --pretrained_model_name_or_path Manojb/stable-diffusion-2-1-base \
  --dataset_name luigi-s/EEG_Image_CVPR_ALL_subj \
  --clip_embeds_dir /content/gwit_runtime/clip_embeds \
  --subject 6 \
  --unclip_ckpt /content/drive/MyDrive/GWIT2/data/unclip6_epoch0_step110000_filtered_for_recon.safetensors \
  --output_dir /content/output/debug_raw_gtclip_decode_subj6 \
  --model_dtype fp32 \
  --num_steps 38 \
  --max_samples 8

/content/micromamba/envs/gwit/lib/python3.9/site-packages/lightning_fabric/__init__.py:29: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __import__("pkg_resources").declare_namespace(__name__)
No SDP backend available, likely because you are running in pytorch versions < 2.0. In fact, you are using PyTorch 1.13.1+cu117. You might want to consider upgrading.
/content/micromamba/envs/gwit/lib/python3.9/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
[DATA] loading full HF pool...
[DA

In [ ]:
!MPLBACKEND=Agg micromamba run -p "{ENV_PATH}" python debug_compare_clip_tokens.py \
  --dataset_name luigi-s/EEG_Image_CVPR_ALL_subj \
  --clip_embeds_dir /content/gwit_runtime/clip_embeds \
  --subject 6 \
  --sample_local_idx 0 \
  --model_dtype fp16

/content/micromamba/envs/gwit/lib/python3.9/site-packages/lightning_fabric/__init__.py:29: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __import__("pkg_resources").declare_namespace(__name__)
No SDP backend available, likely because you are running in pytorch versions < 2.0. In fact, you are using PyTorch 1.13.1+cu117. You might want to consider upgrading.
[DATA] loading full HF pool...
[DATA] subject=6 | local_idx=0 | global_idx=5
[TOKENS] raw npy shape: (1, 256, 1664)
[IMG] gt_img shape: (1, 3, 512, 512)
[RAM] before embedder init | used=2.45 GB | avail=10.22 GB
open_clip_model.safetensors: 100% 10.2G/10.2G [02:27<00:00, 68.7MB/s]
[RAM] after embedder init | used=5.03 GB | avail=7.64 GB
[VRAM] after embedder init | allocated=3.57 GB | reserved=3.90 GB
[TOKENS] direct embedder shape: (1, 2

In [ ]:
!MPLBACKEND=Agg micromamba run -p "{ENV_PATH}" python recon_stageB_unclip.py \
  --stageA_dir /content/drive/MyDrive/GWIT2/output/recon_stageA_subj6 \
  --output_dir /content/output/debug_recon_stageB_subj6 \
  --unclip_ckpt /content/drive/MyDrive/GWIT2/data/unclip6_epoch0_step110000_filtered_for_recon_fp16.pt \
  --unclip_config /content/drive/MyDrive/GWIT2/models/generative_models/configs/unclip6.yaml \
  --num_samples_per_image 1 \
  --decode_batch_size 1 \
  --model_dtype fp16 \
  --resume \
  --resume_minibatch \
  --save_final_manifest \
  --save_vis \
  --log_ram \
  --debug_unclip_stats

/content/micromamba/envs/gwit/lib/python3.9/site-packages/lightning_fabric/__init__.py:29: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __import__("pkg_resources").declare_namespace(__name__)
No SDP backend available, likely because you are running in pytorch versions < 2.0. In fact, you are using PyTorch 1.13.1+cu117. You might want to consider upgrading.
[RAM] before prepare_unclip | used=1.48 GB | avail=11.19 GB
[RAM] before OmegaConf.load | used=1.48 GB | avail=11.19 GB
[RAM] before MinimalUnclipEngine init | used=1.48 GB | avail=11.19 GB
No SDP backend available, likely because you are running in pytorch versions < 2.0. In fact, you are using PyTorch 1.13.1+cu117. You might want to consider upgrading.
SpatialTransformer: Found context dims [1664] of depth 1, which does not match the sp

In [ ]:
!MPLBACKEND=Agg micromamba run -p "{ENV_PATH}" python debug_stageA_vs_gt_tokens.py \
  --pretrained_model_name_or_path Manojb/stable-diffusion-2-1-base \
  --dataset_name luigi-s/EEG_Image_CVPR_ALL_subj \
  --data_root /content/drive/MyDrive/GWIT2/data \
  --clip_embeds_dir /content/gwit_runtime/clip_embeds \
  --model_dir /content/drive/MyDrive/GWIT2/output/zebra_stage3 \
  --output_dir /content/output/debug_stageA_vs_gt \
  --test_subjects 6 \
  --batch_size 4 \
  --prior_inference_steps 20 \
  --prior_cond_scale 1.0 \
  --max_test_samples 8 \
  --save_tensors

/content/micromamba/envs/gwit/lib/python3.9/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
[EEG DATASET] Loaded full HF pool: 11940 samples
[EEG DATASET] After subject index map [6]: 1991 samples
[EEG DATASET] Subject 6 | mode=test | kept 1991/1991 | cap=None
[EEG DATASET] Loaded CLIP image embeds for subj6: shape=(1991, 256, 1664) | mode=sequence
[EEG DATASET] Loaded CLIP text embeds for subj6: shape=(1991, 1280)
[EEG DATASET] Final dataset | mode=test | subjects=[6] | total=1991
[EEG DATASET] CLIP image embedding mode: sequence-level
[EEG DATASET] CLIP text embedding dim: 1280
[DATASET] truncated test set to 8 samples
[EEG Backbone Load]
Source: /content/drive/MyDrive/GWIT

In [ ]:
!micromamba run -p "{ENV_PATH}" python debug_prior_tokens.py \
  --pretrained_model_name_or_path Manojb/stable-diffusion-2-1-base \
  --dataset_name luigi-s/EEG_Image_CVPR_ALL_subj \
  --data_root /content/drive/MyDrive/GWIT2/data \
  --model_dir /content/drive/MyDrive/GWIT2/output/zebra_stage3_v2 \
  --clip_embeds_dir /content/gwit_runtime/clip_embeds \
  --test_subjects 6 \
  --val_ratio 0.1 \
  --split_seed 42 \
  --batch_size 16 \
  --prior_inference_steps 20 \
  --prior_cond_scale 1.0 \
  --max_test_samples 5 \
  --save_per_sample \
  --output_json /content/debug_prior_tokens_test_subj6.json

/content/micromamba/envs/gwit/lib/python3.9/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
[EEG DATASET] Loaded full HF pool: 11940 samples
[EEG DATASET] After subject index map [6]: 1991 samples
[EEG DATASET] Subject 6 | mode=test | kept 1991/1991 | cap=None
[EEG DATASET] Loaded CLIP image embeds for subj6: shape=(1991, 256, 1664) | mode=sequence
[EEG DATASET] Loaded CLIP text embeds for subj6: shape=(1991, 1280)
[EEG DATASET] Final dataset | mode=test | subjects=[6] | total=1991
[EEG DATASET] CLIP image embedding mode: sequence-level
[EEG DATASET] CLIP text embedding dim: 1280
[DATASET] truncated test set to 5 samples
[EEG Backbone Load]
Source: /content/drive/MyDrive/GWIT

In [ ]:
!micromamba run -p "{ENV_PATH}" python debug_make_gt_vs_prior_shards.py \
  --pretrained_model_name_or_path Manojb/stable-diffusion-2-1-base \
  --dataset_name luigi-s/EEG_Image_CVPR_ALL_subj \
  --data_root /content/drive/MyDrive/GWIT2/data \
  --model_dir /content/drive/MyDrive/GWIT2/output/zebra_stage3_v2 \
  --output_dir /content/debug_gt_vs_prior_subj6 \
  --test_subjects 6 \
  --batch_size 16 \
  --num_workers 0 \
  --prior_inference_steps 20 \
  --prior_cond_scale 1.0 \
  --max_test_samples 16 \
  --seed 42

/content/micromamba/envs/gwit/lib/python3.9/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
[EEG DATASET] Loaded full HF pool: 11940 samples
[EEG DATASET] After subject index map [6]: 1991 samples
[EEG DATASET] Subject 6 | mode=test | kept 1991/1991 | cap=None
[EEG DATASET] Loaded CLIP image embeds for subj6: shape=(1991, 256, 1664) | mode=sequence
[EEG DATASET] Loaded CLIP text embeds for subj6: shape=(1991, 1280)
[EEG DATASET] Final dataset | mode=test | subjects=[6] | total=1991
[EEG DATASET] CLIP image embedding mode: sequence-level
[EEG DATASET] CLIP text embedding dim: 1280
[DATASET] truncated test set to 16 samples
[EEG Backbone Load]
Source: /content/drive/MyDrive/GWI

In [ ]:
!micromamba run -p "{ENV_PATH}" python recon_stageA_token_debug_gt_vs_prior.py \
  --pretrained_model_name_or_path Manojb/stable-diffusion-2-1-base \
  --dataset_name luigi-s/EEG_Image_CVPR_ALL_subj \
  --data_root /content/drive/MyDrive/GWIT2/data \
  --clip_embeds_dir /content/gwit_runtime/clip_embeds \
  --model_dir /content/drive/MyDrive/GWIT2/output/zebra_stage3_v3 \
  --output_dir /content/debug_stageA_for_phaseB_subj6 \
  --test_subjects 6 \
  --batch_size 16 \
  --num_workers 0 \
  --prior_inference_steps 20 \
  --prior_cond_scale 1.0 \
  --max_test_samples 16 \
  --seed 42

/content/micromamba/envs/gwit/lib/python3.9/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
[EEG DATASET] Loaded full HF pool: 11940 samples
[EEG DATASET] After subject index map [6]: 1991 samples
[EEG DATASET] Subject 6 | mode=test | kept 1991/1991 | cap=None
[EEG DATASET] Loaded CLIP image embeds for subj6: shape=(1991, 256, 1664) | mode=sequence
[EEG DATASET] Loaded CLIP text embeds for subj6: shape=(1991, 1280)
[EEG DATASET] Final dataset | mode=test | subjects=[6] | total=1991
[EEG DATASET] CLIP image embedding mode: sequence-level
[EEG DATASET] CLIP text embedding dim: 1280
[DATASET] truncated test set to 16 samples
[EEG Backbone Load]
Source: /content/drive/MyDrive/GWI

In [ ]:
!micromamba run -p "{ENV_PATH}" python compare_prior_vs_gt_shards.py \
  --shards_root /content/debug_stageA_for_phaseB_subj1

Comparing shards: 100% 1/1 [00:00<00:00,  5.45it/s]

===== PRIOR vs GT TOKENS =====
num_samples        : 16
mse_mean           : 1.038526
cosine_token_mean  : 0.433131
cosine_pool_mean   : 0.759866
pred_norm_mean     : 26.542620
gt_norm_mean       : 45.417393
[SAVED] /content/debug_gt_vs_prior_subj6/prior_vs_gt_metrics.json


In [ ]:
!MPLBACKEND=Agg micromamba run -p "{ENV_PATH}" python recon_stageB_unclip.py \
  --stageA_dir /content/debug_stageA_for_phaseB_subj6 \
  --output_dir /content/recon_stageB_subj6 \
  --unclip_ckpt /content/drive/MyDrive/GWIT2/data/unclip6_epoch0_step110000_filtered_for_recon.safetensors \
  --unclip_config /content/drive/MyDrive/GWIT2/models/generative_models/configs/unclip6.yaml \
  --num_samples_per_image 1 \
  --decode_batch_size 1 \
  --model_dtype fp32 \
  --resume \
  --resume_minibatch \
  --save_final_manifest \
  --save_vis \
  --log_ram

/content/micromamba/envs/gwit/lib/python3.9/site-packages/lightning_fabric/__init__.py:29: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __import__("pkg_resources").declare_namespace(__name__)
No SDP backend available, likely because you are running in pytorch versions < 2.0. In fact, you are using PyTorch 1.13.1+cu117. You might want to consider upgrading.
[RAM] before prepare_unclip | used=1.51 GB | avail=11.17 GB
[RAM] before OmegaConf.load | used=1.51 GB | avail=11.17 GB
[RAM] before MinimalUnclipEngine init | used=1.51 GB | avail=11.16 GB
No SDP backend available, likely because you are running in pytorch versions < 2.0. In fact, you are using PyTorch 1.13.1+cu117. You might want to consider upgrading.
SpatialTransformer: Found context dims [1664] of depth 1, which does not match the sp